In [5]:
import os
import json
import pandas as pd

# Define paths matching your repository structure
RAW_DIR = "01_data/raw"
PROCESSED_DIR = "01_data/processed"
PYTHON_DIR = "04_python"

# Ensure ALL output directories exist
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(PYTHON_DIR, exist_ok=True)

# 1. Load both files
ledger_df = pd.read_csv(os.path.join(RAW_DIR, "ledger.csv"))
gateway_df = pd.read_csv(os.path.join(RAW_DIR, "gateway.csv"))

print(f"Loaded {len(ledger_df)} ledger records and {len(gateway_df)} gateway records.")

# 2. Check duplicates and nulls
print("\n=== Data Quality Checks ===")
print(f"Ledger Missing Values:\n{ledger_df.isnull().sum()}\n")
print(f"Gateway Missing Values:\n{gateway_df.isnull().sum()}\n")

ledger_dups = ledger_df[ledger_df.duplicated(subset=['transaction_id'], keep=False)]
gateway_dups = gateway_df[gateway_df.duplicated(subset=['transaction_id'], keep=False)]
print(f"Duplicate transaction IDs in Ledger: {ledger_dups['transaction_id'].nunique()}")
print(f"Duplicate transaction IDs in Gateway: {gateway_dups['transaction_id'].nunique()}")

# 3. Outer merge to align records (Updated suffixes to handle 'amount_usd')
merged_df = pd.merge(
    ledger_df,
    gateway_df,
    on='transaction_id',
    how='outer',
    suffixes=('_ledger', '_gateway')
)

# 4. Identify records missing in gateway
missing_in_gateway_mask = merged_df['amount_usd_gateway'].isnull() & merged_df['amount_usd_ledger'].notnull()
missing_in_gateway = ledger_df[ledger_df['transaction_id'].isin(merged_df[missing_in_gateway_mask]['transaction_id'])]
missing_in_gateway.to_csv(os.path.join(PROCESSED_DIR, "missing_in_gateway.csv"), index=False)

# 5. Identify records missing in ledger
missing_in_ledger_mask = merged_df['amount_usd_ledger'].isnull() & merged_df['amount_usd_gateway'].notnull()
missing_in_ledger = gateway_df[gateway_df['transaction_id'].isin(merged_df[missing_in_ledger_mask]['transaction_id'])]
missing_in_ledger.to_csv(os.path.join(PROCESSED_DIR, "missing_in_ledger.csv"), index=False)

# Isolate fully matched transactions to investigate data discrepancies
matched_records = merged_df[merged_df['amount_usd_ledger'].notnull() & merged_df['amount_usd_gateway'].notnull()].copy()

# 6. Identify amount mismatches
amount_mismatches = matched_records[matched_records['amount_usd_ledger'].round(2) != matched_records['amount_usd_gateway'].round(2)]
amount_mismatches.to_csv(os.path.join(PROCESSED_DIR, "amount_mismatches.csv"), index=False)

# 7. Identify status mismatches
status_mismatches = matched_records[
    matched_records['status_ledger'].astype(str).str.lower().str.strip() !=
    matched_records['status_gateway'].astype(str).str.lower().str.strip()
]
status_mismatches.to_csv(os.path.join(PROCESSED_DIR, "status_mismatches.csv"), index=False)

# 8. Build a final reconciliation report
def determine_recon_status(row):
    if pd.isnull(row['amount_usd_ledger']):
        return "Missing in Ledger"
    elif pd.isnull(row['amount_usd_gateway']):
        return "Missing in Gateway"
    elif round(row['amount_usd_ledger'], 2) != round(row['amount_usd_gateway'], 2):
        return "Amount Mismatch"
    elif str(row['status_ledger']).lower().strip() != str(row['status_gateway']).lower().strip():
        return "Status Mismatch"
    else:
        return "Reconciled"

merged_df['reconciliation_status'] = merged_df.apply(determine_recon_status, axis=1)
merged_df.to_csv(os.path.join(PROCESSED_DIR, "reconciliation_report.csv"), index=False)

# 9. Generate summary metrics
unreconciled_mask = merged_df['reconciliation_status'] != 'Reconciled'
unreconciled_df = merged_df[unreconciled_mask]

ledger_risk = unreconciled_df['amount_usd_ledger'].fillna(0).sum()
gateway_only_risk = unreconciled_df[unreconciled_df['amount_usd_ledger'].isnull()]['amount_usd_gateway'].fillna(0).sum()
amount_at_risk = ledger_risk + gateway_only_risk

summary_metrics = {
    "total_ledger_rows": int(len(ledger_df)),
    "total_gateway_rows": int(len(gateway_df)),
    "missing_in_gateway_count": int(missing_in_gateway_mask.sum()),
    "missing_in_ledger_count": int(missing_in_ledger_mask.sum()),
    "amount_mismatch_count": int(len(amount_mismatches)),
    "status_mismatch_count": int(len(status_mismatches)),
    "reconciliation_issue_count": int(unreconciled_mask.sum()),
    "ledger_total_amount": float(ledger_df['amount_usd'].sum()),
    "gateway_total_amount": float(gateway_df['amount_usd'].sum()),
    "amount_at_risk": float(amount_at_risk)
}

# Write summary metrics to JSON
with open(os.path.join(PYTHON_DIR, "summary_metrics.json"), "w") as f:
    json.dump(summary_metrics, f, indent=4)

print("\n=== Pipeline Execution Completed Successfully ===")
print(json.dumps(summary_metrics, indent=4))

Loaded 10 ledger records and 9 gateway records.

=== Data Quality Checks ===
Ledger Missing Values:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

Gateway Missing Values:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

Duplicate transaction IDs in Ledger: 0
Duplicate transaction IDs in Gateway: 0

=== Pipeline Execution Completed Successfully ===
{
    "total_ledger_rows": 10,
    "total_gateway_rows": 9,
    "missing_in_gateway_count": 2,
    "missing_in_ledger_count": 1,
    "amount_mismatch_count": 2,
    "status_mismatch_count": 1,
    "reconciliation_issue_count": 6,
    "ledger_total_amount": 23340.0,
    "gateway_total_amount": 20550.0,
    "amount_at_risk": 15090.0
}


In [6]:
import os
import json
import pandas as pd

# Define your directory paths
PROCESSED_DIR = "01_data/processed"
PYTHON_DIR = "04_python"

print("=== Starting Dashboard Source Preparation ===")

# 1. Load and Normalize JSON Summary Metrics
json_path = os.path.join(PYTHON_DIR, "summary_metrics.json")
with open(json_path, "r") as f:
    metrics_data = json.load(f)

# pd.json_normalize flattens nested structures into a single-row DataFrame row
metrics_df = pd.json_normalize(metrics_data)
print("\nNormalized JSON Metrics Structure:")
print(metrics_df.to_string(index=False))

# 2. Load the main Reconciliation Report
recon_report_path = os.path.join(PROCESSED_DIR, "reconciliation_report.csv")
recon_df = pd.read_csv(recon_report_path)

# 3. Create a Consolidated Dashboard Source
# Broadcasting summary metrics across all rows makes it incredibly easy
# for dashboard filters to calculate totals dynamically alongside transaction rows.
dashboard_ready_df = recon_df.copy()
for column in metrics_df.columns:
    dashboard_ready_df[f"meta_{column}"] = metrics_df[column].iloc[0]

# 4. Save the final dashboard source output
dashboard_output_path = os.path.join(PROCESSED_DIR, "dashboard_source.csv")
dashboard_ready_df.to_csv(dashboard_output_path, index=False)

print(f"\n[SUCCESS] Dashboard source output generated with shape: {dashboard_ready_df.shape}")
print(f"Saved to: {dashboard_output_path}")

=== Starting Dashboard Source Preparation ===

Normalized JSON Metrics Structure:
 total_ledger_rows  total_gateway_rows  missing_in_gateway_count  missing_in_ledger_count  amount_mismatch_count  status_mismatch_count  reconciliation_issue_count  ledger_total_amount  gateway_total_amount  amount_at_risk
                10                   9                         2                        1                      2                      1                           6              23340.0               20550.0         15090.0

[SUCCESS] Dashboard source output generated with shape: (11, 22)
Saved to: 01_data/processed/dashboard_source.csv
